<h1 aling=center font-family: Helvetica>Extrator de Estaçoes (extraindo HTML) </h1>

<h3 align=center font-family: Helvetica> Lendo tabela do HTML </h3>

In [1]:
## Importando pandas
import pandas as pd

## concertando UTF
with open("../Extraindo_Estacoes/estacoes.html", "r", encoding="utf-8") as arquivo:
    html = arquivo.read()

## Lendo HTML com a tabela
tabelas = pd.read_html(html)

## Mostrar tabela
print(tabelas)

[                                       0      1     2
0                                Estação  Linha  Zona
1                               Abbesses     12     1
2                                 Alésia      4     1
3                        Alexandre Dumas      2     1
4                         Alma – Marceau      9     1
..                                   ...    ...   ...
295             Villejuif – Louis Aragon      7     2
296  Villejuif – Paul Vaillant-Couturier      7     2
297                             Villiers  2 & 3     1
298                          Volontaires     12     1
299                             Voltaire      9     1

[300 rows x 3 columns]]


C:\Users\joaoe\AppData\Local\Temp\ipykernel_5536\697251729.py:9: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabelas = pd.read_html(html)


<h3 align=center font-family: Helvetica> Excluindo campo desnecessario</h3>

In [2]:

## Seleciona a tabela
TabelaSelecionada = tabelas[0]

## Vendo nomes das colunas
print(TabelaSelecionada.columns)

## excluindo zona da tabela
TabelaSelecionada = TabelaSelecionada.drop(2, axis = 1)

## mostrando depois da alteracao
print(TabelaSelecionada)

Index([0, 1, 2], dtype='int64')
                                       0      1
0                                Estação  Linha
1                               Abbesses     12
2                                 Alésia      4
3                        Alexandre Dumas      2
4                         Alma – Marceau      9
..                                   ...    ...
295             Villejuif – Louis Aragon      7
296  Villejuif – Paul Vaillant-Couturier      7
297                             Villiers  2 & 3
298                          Volontaires     12
299                             Voltaire      9

[300 rows x 2 columns]


<h3 align=center font-family: Helvetica> Renomeando colunas/linhas </h3>

In [3]:
TabelaSelecionada.columns = ["Estacao", "Linha"]
TabelaSelecionada = TabelaSelecionada.drop(index=0)
print(TabelaSelecionada)

                                 Estacao  Linha
1                               Abbesses     12
2                                 Alésia      4
3                        Alexandre Dumas      2
4                         Alma – Marceau      9
5                         Anatole France      3
..                                   ...    ...
295             Villejuif – Louis Aragon      7
296  Villejuif – Paul Vaillant-Couturier      7
297                             Villiers  2 & 3
298                          Volontaires     12
299                             Voltaire      9

[299 rows x 2 columns]


<h3 align=center font-family:Helvetica> Salvando valores para utilziar depois </h3>

In [4]:
## TabelaSelecionada.to_json('arquivo.json', orient='records', indent=4, force_ascii=False)

<h3 align=center font-family:Helvetica> Concertando os valores do dataframe</h3>

In [5]:
import re

for linha in TabelaSelecionada.itertuples():
    TabelaSelecionada.at[linha.Index, "Linha"] = re.split(
        r"\s*[&,]\s*", linha.Linha
    )
    
print(TabelaSelecionada)

                                 Estacao   Linha
1                               Abbesses    [12]
2                                 Alésia     [4]
3                        Alexandre Dumas     [2]
4                         Alma – Marceau     [9]
5                         Anatole France     [3]
..                                   ...     ...
295             Villejuif – Louis Aragon     [7]
296  Villejuif – Paul Vaillant-Couturier     [7]
297                             Villiers  [2, 3]
298                          Volontaires    [12]
299                             Voltaire     [9]

[299 rows x 2 columns]


<h3 align=center font-family:Helvetica> Extraçao das estaçoes presente nas linhas </h3>

In [6]:
import json

## As estaçoes que serao utilizadas serao as que estiverem nessa linha
## N=1: 1,2,3,4,6,9,11,12,13,14

## linhas obrigatorias
linhasObrigatorias = ['1','2','3','4','6','9','11','12','13','14']

estacoesEncontradas = []

## estaçoes que estao nas linhas
for itens in TabelaSelecionada.itertuples():
    if any(numero in itens.Linha for numero in linhasObrigatorias):
        estacoesEncontradas.append({
            "Estacao": itens.Estacao,
            "Linha": itens.Linha
        })
        
## Lista das estacoes
print(len(estacoesEncontradas))

## resultado (Para ver localmente)
jsonEstacoes = json.dumps(
    estacoesEncontradas,
    ensure_ascii=False,
    indent=1
)

## salvando
## with open('EstacoesLinhas.json', 'w', encoding='utf-8') as arquivo:
    ## json.dump(estacoesEncontradas, arquivo, ensure_ascii=False, indent=4)

207


<h3 align=center font-family:Helvetica> Extraçao das estaçoes que serao utilizadas </h3>

In [ ]:
## Import unicodedata
import unicodedata

## Inicio e fim de cada linha

# Linha 1: La Défense ↔ Château de Vincennes (cor amarela)
# Linha 2: Porte Dauphine ↔ Nation (cor azul)
# Linha 3: Pont de Levallois – Bécon ↔ Gallieni (cor oliva)
# Linha 4: Porte de Clignancourt ↔ Bagneux – Lucie Aubrac (cor roxa)
# Linha 6: Charles de Gaulle – Étoile ↔ Nation (cor verde-clara)
# Linha 9: Pont de Sèvres ↔ Mairie de Montreuil (cor ocre-amarela)
# Linha 11: Châtelet ↔ Rosny-Bois-Perrier (cor marrom)
# Linha 12: Mairie d'Aubervilliers ↔ Mairie d'Issy (cor verde)
# Linha 13: Saint-Denis – Université / Asnières – Gennevilliers – Les Courtilles ↔ Châtillon – Montrouge (cor azul-clara)
# Linha 14: Saint-Denis – Pleyel ↔ Aéroport d'Orly (cor roxa/violeta)

linhasMetro = {
    1: {
        "inicio": "La Défense",
        "fim": "Château de Vincennes"
    },
    2: {
        "inicio": "Porte Dauphine",
        "fim": "Nation"
    },
    3: {
        "inicio": "Pont de Levallois – Bécon",
        "fim": "Gallieni"
    },
    4: {
        "inicio": "Porte de Clignancourt",
        "fim": "Bagneux – Lucie Aubrac"
    },
    6: {
        "inicio": "Charles de Gaulle – Étoile",
        "fim": "Nation"
    },
    9: {
        "inicio": "Pont de Sèvres",
        "fim": "Mairie de Montreuil"
    },
    11: {
        "inicio": "Châtelet", 
        "fim": "Mairie des Lilas"
    },
    12: {"inicio": "Porte de la Chapelle", 
        "fim": "Mairie d'Issy"
    },
    13: {
        "inicio": ["Saint-Denis – Université", "Asnières – Gennevilliers – Les Courtilles"],
        "fim": "Châtillon – Montrouge"
    },
    14: {"inicio": "Saint-Lazare", 
        "fim": "Olympiades"
    }
}


## Evita erro de comparação por acentos codificados de forma diferente
def normalizar(nome):
    return unicodedata.normalize("NFC", str(nome)).strip()

## Conjunto com todas as estações terminais
terminais = set()
for numeroLinha, extremos in linhasMetro.items():
    for chave in ("inicio", "fim"):
        valor = extremos[chave]
        nomes = valor if isinstance(valor, list) else [valor]
        for nome in nomes:
            terminais.add(normalizar(nome))

In [8]:
## Estações que entram: servem 2+ linhas do grupo OU são início/fim de linha
estacoesSelecionadas = []
nomesNaTabela = set()

for item in estacoesEncontradas:
    nome = normalizar(item["Estacao"])
    nomesNaTabela.add(nome)

    ## Só conta as linhas do grupo (3bis, 7bis e outras são ignoradas)
    linhasDoGrupo = [l for l in item["Linha"] if l in linhasObrigatorias]

    if len(set(linhasDoGrupo)) >= 2:
        estacoesSelecionadas.append({"Estacao": item["Estacao"], "Linha": linhasDoGrupo})
    elif nome in terminais:
        estacoesSelecionadas.append({"Estacao": item["Estacao"], "Linha": linhasDoGrupo})

print("Estações selecionadas:", len(estacoesSelecionadas))

## Terminais que NÃO existem na tabela do HTML
ausentes = sorted(terminais - nomesNaTabela)
print("Terminais ausentes da tabela:", ausentes)

## Salvando
with open('EstacoesSelecionadas.json', 'w', encoding='utf-8') as arquivo:
    json.dump(estacoesSelecionadas, arquivo, ensure_ascii=False, indent=4)

Estações selecionadas: 43
Terminais ausentes da tabela: ['Bagneux – Lucie Aubrac', 'La Défense']
